# Calendar Dimension Table

Builds the standard calendar dimension `ws_healthcare_cntrl_india.silver.calendar` with one row per day from 1940-01-01 to 2030-12-31, including calendar, ISO week, and Indian fiscal year (April-March) attributes.

In [0]:
CREATE OR REPLACE TABLE ws_healthcare_cntrl_india.silver.calendar
COMMENT 'Standard calendar dimension: one row per day from 1940-01-01 to 2030-12-31. Fiscal year follows Indian FY (April-March).'
AS
WITH dates AS (
  SELECT explode(sequence(DATE'1940-01-01', DATE'2030-12-31', INTERVAL 1 DAY)) AS date
)
SELECT
  CAST(date_format(date, 'yyyyMMdd') AS INT)                         AS date_key,
  date                                                               AS date,

  -- Day attributes
  day(date)                                                          AS day_of_month,
  dayofyear(date)                                                    AS day_of_year,
  weekday(date) + 1                                                  AS day_of_week,        -- 1 = Monday ... 7 = Sunday (ISO)
  dayofweek(date)                                                    AS day_of_week_sun_start, -- 1 = Sunday ... 7 = Saturday
  date_format(date, 'EEEE')                                          AS day_name,
  date_format(date, 'EEE')                                           AS day_name_short,
  CASE WHEN weekday(date) >= 5 THEN TRUE ELSE FALSE END              AS is_weekend,
  CASE WHEN weekday(date) < 5 THEN TRUE ELSE FALSE END               AS is_weekday,

  -- Week attributes (ISO 8601: weeks start Monday)
  weekofyear(date)                                                   AS week_of_year,
  year(date_add(date, 3 - weekday(date)))                            AS iso_year,
  concat(year(date_add(date, 3 - weekday(date))), '-W', lpad(weekofyear(date), 2, '0')) AS iso_year_week,
  CAST(ceil(day(date) / 7.0) AS INT)                                 AS week_of_month,
  date_sub(date, weekday(date))                                      AS week_start_date,
  date_add(date, 6 - weekday(date))                                  AS week_end_date,

  -- Month attributes
  month(date)                                                        AS month,
  date_format(date, 'MMMM')                                          AS month_name,
  date_format(date, 'MMM')                                           AS month_name_short,
  date_format(date, 'yyyy-MM')                                       AS year_month,
  CAST(date_format(date, 'yyyyMM') AS INT)                           AS year_month_key,
  trunc(date, 'MM')                                                  AS month_start_date,
  last_day(date)                                                     AS month_end_date,
  day(last_day(date))                                                AS days_in_month,
  date = trunc(date, 'MM')                                           AS is_month_start,
  date = last_day(date)                                              AS is_month_end,

  -- Quarter attributes
  quarter(date)                                                      AS quarter,
  concat('Q', quarter(date))                                         AS quarter_name,
  concat(year(date), '-Q', quarter(date))                            AS year_quarter,
  trunc(date, 'QUARTER')                                             AS quarter_start_date,
  date_sub(add_months(trunc(date, 'QUARTER'), 3), 1)                 AS quarter_end_date,
  datediff(date, trunc(date, 'QUARTER')) + 1                         AS day_of_quarter,

  -- Half-year attributes
  CASE WHEN month(date) <= 6 THEN 1 ELSE 2 END                       AS half_year,
  concat(year(date), '-H', CASE WHEN month(date) <= 6 THEN 1 ELSE 2 END) AS year_half,

  -- Year attributes
  year(date)                                                         AS year,
  trunc(date, 'YEAR')                                                AS year_start_date,
  make_date(year(date), 12, 31)                                      AS year_end_date,
  CASE WHEN (year(date) % 4 = 0 AND year(date) % 100 <> 0) OR year(date) % 400 = 0
       THEN TRUE ELSE FALSE END                                      AS is_leap_year,
  CAST(floor(year(date) / 10) * 10 AS INT)                           AS decade,

  -- Indian fiscal year (April 1 - March 31); fiscal_year = year in which FY ends
  CASE WHEN month(date) >= 4 THEN year(date) + 1 ELSE year(date) END AS fiscal_year,
  CASE WHEN month(date) >= 4
       THEN concat('FY', year(date), '-', right(CAST(year(date) + 1 AS STRING), 2))
       ELSE concat('FY', year(date) - 1, '-', right(CAST(year(date) AS STRING), 2)) END AS fiscal_year_name,
  CAST(floor(((month(date) + 8) % 12) / 3) + 1 AS INT)               AS fiscal_quarter,
  ((month(date) + 8) % 12) + 1                                       AS fiscal_month,

  -- Relative flags (evaluated at load time)
  date = current_date()                                              AS is_today,
  datediff(date, current_date())                                     AS days_from_today
FROM dates;

In [0]:
SELECT
  count(*)                         AS total_rows,
  count(DISTINCT date_key)         AS distinct_date_keys,
  min(date)                        AS min_date,
  max(date)                        AS max_date,
  datediff(max(date), min(date)) + 1 AS expected_rows,
  sum(CASE WHEN is_leap_year AND month = 2 AND day_of_month = 29 THEN 1 ELSE 0 END) AS leap_days
FROM ws_healthcare_cntrl_india.silver.calendar;

In [0]:
SELECT *
FROM ws_healthcare_cntrl_india.silver.calendar
WHERE date IN (DATE'1940-01-01', DATE'2024-02-29', DATE'2024-03-31', DATE'2024-04-01', DATE'2026-12-31', DATE'2030-12-31')
ORDER BY date;